# TP 21 - Orchestration et outils : l'assistant NutriScope

## Objectif

Construire un assistant conversationnel multi-tours. Il répond aux questions d'un utilisateur sur un produit alimentaire et choisit lui-même l'outil à appeler : chercher le produit dans la base, proposer un produit de substitution mieux noté, ou expliquer en citant une source.

## Architecture

| Brique | Outil | Rôle |
|---|---|---|
| Agent, mémoire, outils | LangChain | fait tourner la boucle de l'agent et garde l'historique de chaque conversation |
| Modèle de chat | "ChatOpenAI" branché sur Ollama | comprend la question, choisit l'outil, rédige la réponse |
| Recherche documentaire | LlamaIndex, vecteurs stockés dans Chroma | retrouve les passages utiles et leur source |
| Suivi des conversations | journal JSONL et traces MLflow, en local | garde, pour chaque échange, les outils appelés, la durée et les tokens |

## Les outils de l'assistant

| Outil | Ce qu'il fait | Vrai code prévu | État dans ce notebook |
|---|---|---|---|
| "search_product" | trouve un produit par son nom ou son code-barres | TP 18 | placeholder |
| "suggest_substitutes" | propose des produits comparables mieux notés, en bio si demandé | TP 16 | placeholder |
| "answer_with_sources" | retrouve des passages de documents et donne leur source | TP 19-20 | placeholder, puis mini-index LlamaIndex |

Attention : les données renvoyées par les placeholders sont inventées. Elles respectent le format des vrais outils mais ne doivent jamais servir à une démonstration.

## Hors périmètre

- LangSmith : c'est un service en ligne, les conversations sortiraient de la machine. MLflow fait le même travail en local.
- Le résumé automatique de la mémoire et la confirmation humaine : pas nécessaires pour des conversations de quelques tours.
- La prédiction du Nutri-Score : elle deviendra un quatrième outil quand le modèle ML sera entraîné et sauvegardé.

## Choix des outils

- LangChain plutôt qu'une boucle écrite à la main avec le client "openai" : la boucle de l'agent, la déclaration des outils et la mémoire sont fournies, il reste moins de code à maintenir.
- "ChatOpenAI" plutôt que "ChatOllama" : c'est le même format OpenAI que l'atelier scanner. On passe d'Ollama en local à un fournisseur en ligne en changeant seulement l'adresse et le nom du modèle.
- LlamaIndex pour le RAG : le corpus sera fait de documents bruts de formats variés. LlamaIndex fournit leur lecture, leur découpage et leur réindexation, qu'il faudrait sinon écrire à la main format par format. Je veux aussi maîtriser cet outil pour des utilisations futures.
- MLflow pour tracer les conversations : les traces restent sur la machine, sans compte ni clé. Le même outil servira au suivi des entraînements ML. Il complète le journal JSONL, qui reste le journal rendu.

## Plan du notebook

1. Connexion au modèle
2. LangChain en bref
3. Le RAG avec LlamaIndex
4. Les outils "search_product" et "suggest_substitutes"
5. L'agent avec mémoire courte
6. Journalisation (journal JSONL et traces MLflow)
7. Scénarios rejouables
8. Bilan

## 1. Connexion au modèle

Le modèle de chat est "qwen3:8b", servi en local par Ollama. On l'appelle avec "ChatOpenAI", au format OpenAI, comme dans l'atelier scanner.

La température est à 0 : à question égale, le modèle répond de la même façon, ce qui rend les scénarios rejouables.

In [90]:
import os
import re
from collections import Counter

import chromadb
import duckdb
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from pypdf import PdfReader

from src.config import CLEAN_DIR, CLEAN_PREFIX, DATA_DIR, ROOT
from src.utils import find_input

In [62]:
load_dotenv()  # lit le fichier .env s'il existe

OLLAMA_URL = os.getenv("OLLAMA_URL") or "http://localhost:11434"
CHAT_MODEL = "qwen3:8b"

model = ChatOpenAI(
    base_url=f"{OLLAMA_URL}/v1",  # point d'accès au format OpenAI d'Ollama
    api_key="ollama",             # obligatoire pour le format OpenAI, ignorée par Ollama
    model=CHAT_MODEL,
    temperature=0,
)

In [63]:
response = model.invoke("Réponds en une phrase : qu'est-ce que le Nutri-Score ?")

print(response.content)
print(response.usage_metadata)

Le Nutri-Score est un système de notation des aliments, utilisé en France, qui classe les produits alimentaires sur une échelle de A (le plus sain) à E (le moins sain) en fonction de leur teneur en nutriments et en ingrédients peu bénéfiques pour la santé.
{'input_tokens': 27, 'output_tokens': 266, 'total_tokens': 293, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


## 2. LangChain en bref

Trois briques de base, avant de construire l'agent :

1. les messages : une conversation est une liste de messages, chacun avec un rôle
2. les gabarits de prompt : un texte à trous, rempli au moment de l'appel
3. les chaînes : des briques branchées les unes derrière les autres avec "|"

### 2.1. Les messages

Chaque message a un rôle. "SystemMessage" donne les consignes de l'assistant, "HumanMessage" porte la question de l'utilisateur, et la réponse du modèle est un "AIMessage".

In [64]:
messages = [
    SystemMessage("Tu es l'assistant NutriScope. Réponds en une phrase."),
    HumanMessage("Les chips sont-elles bonnes pour la santé ?"),
]

response = model.invoke(messages)
print(type(response).__name__, ":", response.content)

AIMessage : Les chips sont généralement peu saines en raison de leur teneur élevée en graisses, sel et calories, mais peuvent être consommées en modération ou remplacées par des alternatives plus saines comme les chips croustillantes faites maison.


### 2.2. Les gabarits de prompt

"ChatPromptTemplate" contient des variables entre accolades. Elles sont remplies à l'appel, et une variable oubliée provoque une erreur au lieu d'envoyer un prompt incomplet.

In [65]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es l'assistant NutriScope. Réponds en une phrase."),
    ("human", "Le produit {product} est-il bon pour la santé ?"),
])

print(prompt.invoke({"product": "Nutella"}).to_messages())

[SystemMessage(content="Tu es l'assistant NutriScope. Réponds en une phrase.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Le produit Nutella est-il bon pour la santé ?', additional_kwargs={}, response_metadata={})]


### 2.3. Les chaînes

L'opérateur "|" branche les briques : la sortie de l'une devient l'entrée de la suivante. Ici, le gabarit produit les messages, le modèle répond, et "StrOutputParser" ne garde que le texte de la réponse.

In [66]:
chain = prompt | model | StrOutputParser()

print(chain.invoke({"product": "Nutella"}))

Nutella est un produit sucré et gras, à consommer avec modération en raison de son taux élevé de sucre et de matières grasses, mais il apporte des nutriments comme la fibre et les antioxydants.


### 2.4. Le modèle ne se souvient de rien

Chaque appel est indépendant. Une question comme "Pourquoi ?", posée seule, n'a aucun sens pour le modèle. Pour qu'il suive la conversation, il faut lui renvoyer tout l'historique à chaque appel.

Ici on le fait à la main, avec une liste. Dans la section 5, "InMemorySaver" le fera automatiquement pour l'agent.

In [67]:
# Sans historique : le modèle ne sait pas de quoi on parle
print(model.invoke("Pourquoi ?").content)

Pourquoi ? 😊  
Pourriez-vous préciser de quoi vous parlez ? Je suis là pour vous aider !


In [68]:
# Avec historique : on renvoie toute la conversation à chaque appel
history = [
    SystemMessage("Tu es l'assistant NutriScope. Réponds en une phrase."),
    HumanMessage("Le Nutella est-il bon pour la santé ?"),
]
history.append(model.invoke(history))  # on garde la réponse dans l'historique
history.append(HumanMessage("Pourquoi ?"))

print(model.invoke(history).content)

Le Nutella est riche en sucre et en graisses (notamment en huile de palme), ce qui en fait un aliment calorique et peu nutritif, à consommer avec modération pour éviter les excès et préserver la santé.


## 3. Le RAG avec LlamaIndex

### 3.1. Le principe

Le modèle ne connaît pas nos sources et peut inventer (voir l'exemple du Nutella en 2.3). Le RAG (Retrieval-Augmented Generation, génération augmentée par la recherche) lui fournit des extraits de documents fiables au moment de répondre.

| Étape | Ce qu'elle fait | Quand |
|---|---|---|
| Lire | extraire le texte des pages web et des PDF de "data/corpus/" | une fois, à l'indexation |
| Découper | couper les longs textes en morceaux qui se recouvrent un peu | une fois, à l'indexation |
| Vectoriser | transformer chaque morceau en vecteur avec le modèle d'embeddings | une fois, à l'indexation |
| Ranger | stocker les vecteurs et leurs métadonnées dans Chroma | une fois, à l'indexation |
| Retrouver | vectoriser la question et chercher les morceaux les plus proches | à chaque question |

LlamaIndex s'occupe de ces cinq étapes. Il ne génère aucune réponse : les extraits retrouvés seront donnés à l'agent LangChain, qui rédigera la réponse en citant ses sources.

On indexe deux contenus, dans deux collections séparées : le corpus nutrition d'un collègue (25 sources officielles, 405 morceaux, section 3.5) et des fiches produits construites à partir d'Open Food Facts (section 3.7).


### 3.2. Le modèle d'embeddings

Un embedding est un vecteur de nombres qui représente le sens d'un texte. Deux textes de sens proche donnent des vecteurs proches.

On utilise "qwen3-embedding:0.6b", servi par Ollama. Il est réglé une seule fois dans "Settings", les réglages globaux de LlamaIndex. Les documents et les questions passent ainsi par le même modèle, sinon les distances entre vecteurs n'auraient pas de sens.

Ce modèle attend une consigne devant les questions ("Instruct: ... Query:"), mais pas devant les documents. C'est le rôle de "query_instruction".


In [69]:
EMBED_MODEL = "qwen3-embedding:0.6b"

Settings.embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=OLLAMA_URL,  # sans "/v1" : LlamaIndex utilise l'API d'Ollama, pas le format OpenAI
    query_instruction="Instruct: Retrouver les passages qui répondent à une question sur la nutrition\nQuery:",
)

vector = Settings.embed_model.get_query_embedding("Comment est calculé le Nutri-Score ?")
print("Dimension du vecteur :", len(vector))

Dimension du vecteur : 1024


Les nombres d'un vecteur n'ont pas de sens pris un par un. Ce qui compte, c'est la comparaison entre deux vecteurs, mesurée par la similarité cosinus : proche de 1 pour des sens proches, plus basse pour des textes sans rapport.

On compare la question à quatre phrases : une qui y répond, une fausse qui a la même forme, une qui parle du Nutri-Score sans y répondre, et une sur un autre sujet.


In [70]:
def cosine_similarity(a, b):
    """Calcule la similarité cosinus entre deux vecteurs (1 = même direction)."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


question = "Comment est calculé le Nutri-Score ?"
sentences = [
    "Le Nutri-Score est calculé pour 100 g : l'énergie, les sucres, les graisses saturées et le sel font baisser la note, les fibres, les protéines, les fruits et les légumes la font monter.",
    "Le Nutri-Score est calculé à partir du prix du produit et de sa marque.",  # phrase fausse
    "Le Nutri-Score est affiché sur les emballages de façon volontaire par les fabricants.",
    "Il est recommandé de limiter la charcuterie à 150 g par semaine.",
]

question_vector = Settings.embed_model.get_query_embedding(question)  # avec la consigne

for sentence in sentences:
    sentence_vector = Settings.embed_model.get_text_embedding(sentence)  # sans la consigne
    score = cosine_similarity(question_vector, sentence_vector)
    print(f"{score:.2f}  {sentence[:70]}...")

0.86  Le Nutri-Score est calculé pour 100 g : l'énergie, les sucres, les gra...
0.78  Le Nutri-Score est calculé à partir du prix du produit et de sa marque...
0.63  Le Nutri-Score est affiché sur les emballages de façon volontaire par ...
0.28  Il est recommandé de limiter la charcuterie à 150 g par semaine....


La phrase fausse obtient 0,78, presque autant que la bonne réponse (0,86) : elle a le même sujet et la même forme. La similarité mesure la proximité de sens, pas la vérité. D'où l'importance d'un corpus de sources fiables, de plusieurs extraits renvoyés (top-k) et d'une mesure de la recherche.


### 3.3. La lecture du corpus

Les 5 documents ont été enregistrés à la main dans "data/corpus/" (hors Git), chacun nommé par son id. Le site de Vie publique bloque les téléchargements automatiques, et des fichiers locaux rendent le notebook reproductible.

Deux fonctions extraient le texte brut :

- "read_html" ne garde que la balise "main" d'une page web, c'est-à-dire le contenu, sans le menu ni le pied de page (beautifulsoup4),
- "read_pdf" assemble le texte de toutes les pages d'un PDF (pypdf).

Chaque texte devient un "Document" LlamaIndex, accompagné de ses métadonnées : titre, éditeur, URL, date de consultation et licence. Elles serviront à citer la source dans les réponses.

La liste complète des sources et de leurs licences sera tenue dans "docs/ia/corpus.md".


In [71]:
CORPUS_DIR = DATA_DIR / "corpus"

# # Métadonnées des sources : la clé est l'id, qui est aussi le nom du fichier
# SOURCES = {
#     "viepub-ns": {
#         "titre": "Le Nutri-Score en cinq questions",
#         "editeur": "Vie publique (DILA)",
#         "url": "https://www.vie-publique.fr/questions-reponses/286134-le-nutri-score-en-cinq-questions",
#         "fichier": "viepub-ns.html",
#         "licence": "Licence Ouverte 2.0",
#     },
#     "santefr-ns": {
#         "titre": "Ce qu'il faut savoir sur le nouveau Nutri-Score",
#         "editeur": "Santé.fr",
#         "url": "https://www.sante.fr/ce-quil-faut-savoir-sur-le-nouveau-nutri-score",
#         "fichier": "santefr-ns.html",
#         "licence": "CGU de Santé.fr, à vérifier",
#     },
#     "spf-ns": {
#         "titre": "Nutri-Score",
#         "editeur": "Santé publique France",
#         "url": "https://www.santepubliquefrance.fr/determinants-de-sante/nutrition-et-activite-physique/articles/nutri-score",
#         "fichier": "spf-ns.html",
#         "licence": "mentions légales de Santé publique France, à vérifier",
#     },
#     "pnns-adultes": {
#         "titre": "Recommandations relatives à l'alimentation, à l'activité physique et à la sédentarité pour les adultes",
#         "editeur": "Santé publique France",
#         "url": "https://www.santepubliquefrance.fr/sites/default/files/rdd/document/42466_spf00000619.pdf",
#         "fichier": "pnns-adultes.pdf",
#         "licence": "mentions légales de Santé publique France, à vérifier",
#     },
#     "hcsp-2017": {
#         "titre": "Avis relatif à la révision des repères alimentaires pour les adultes",
#         "editeur": "Haut Conseil de la santé publique (HCSP)",
#         "url": "https://www.hcsp.fr/Explore.cgi/Telecharger?NomFichier=hcspa20170216_reperesalimentairesactua2017.pdf",
#         "fichier": "hcsp-2017.pdf",
#         "licence": "à vérifier",
#     },
# }
# CONSULTED_ON = "2026-11-24"  # date d'enregistrement des fichiers

In [72]:
# def read_html(path):
#     """Rend le texte de la balise "main" d'une page web enregistrée."""
#     with open(path, encoding="utf-8") as file:
#         soup = BeautifulSoup(file, "html.parser")
#     main = soup.find("main")
#     return main.get_text(separator="\n", strip=True)


# def read_pdf(path):
#     """Rend le texte de toutes les pages d'un PDF, mises bout à bout."""
#     reader = PdfReader(path)
#     pages = []
#     for page in reader.pages:
#         pages.append(page.extract_text())
#     return "\n".join(pages)

In [73]:
# documents = []
# for source_id, source in SOURCES.items():
#     path = CORPUS_DIR / source["fichier"]
#     if path.suffix == ".html":
#         text = read_html(path)
#     else:
#         text = read_pdf(path)

#     document = Document(
#         text=text,
#         metadata={
#             "source_id": source_id,
#             "titre": source["titre"],
#             "editeur": source["editeur"],
#             "url": source["url"],
#             "consulte_le": CONSULTED_ON,
#             "licence": source["licence"],
#         },
#         # ces métadonnées ne doivent pas être vectorisées avec le texte : elles ne parlent pas de nutrition
#         excluded_embed_metadata_keys=["url", "consulte_le", "licence"],
#     )
#     documents.append(document)

# for document in documents:
#     print(f"{document.metadata['source_id']:13} {len(document.text):7} caractères | {document.text[:60]!r}")

Cellule de contrôle : avant de découper, on vérifie la qualité du texte extrait. Elle écrit chaque texte dans un fichier ".txt" de "data/corpus/", à ouvrir dans VS Code à côté de l'original. Elle signale aussi les pages de PDF sans texte (une page scannée est une image, pypdf n'en tire rien) et les lignes qui se répètent souvent (en-têtes, pieds de page, menus).


In [74]:
# for document in documents:
#     source_id = document.metadata["source_id"]

#     # 1. Le texte extrait, dans un fichier à ouvrir dans VS Code
#     txt_path = CORPUS_DIR / f"{source_id}.txt"
#     txt_path.write_text(document.text, encoding="utf-8")
#     print(f"=== {source_id} : {len(document.text)} caractères, écrit dans {txt_path.name}")

#     # 2. Pour un PDF : les pages sans texte
#     if SOURCES[source_id]["fichier"].endswith(".pdf"):
#         reader = PdfReader(CORPUS_DIR / SOURCES[source_id]["fichier"])
#         empty_pages = []
#         for number, page in enumerate(reader.pages, start=1):
#             if page.extract_text().strip() == "":
#                 empty_pages.append(number)
#         print(f"    {len(reader.pages)} pages, sans texte : {empty_pages}")

#     # 3. Les lignes répétées au moins 3 fois : en-têtes, pieds de page, menus
#     line_counts = Counter(line.strip() for line in document.text.split("\n") if line.strip())
#     for line, count in line_counts.most_common(5):
#         if count >= 3:
#             print(f"    {count:3} fois : {line[:70]!r}")

### 3.4. Le découpage

Un document entier est trop long pour être un bon extrait : son vecteur mélangerait tous ses sujets, et l'agent recevrait des pages entières à lire. On le coupe donc en morceaux, que LlamaIndex appelle des nœuds ("nodes").

"SentenceSplitter" coupe en respectant les phrases et les paragraphes, pour ne pas couper une phrase au milieu. Deux réglages :

| Réglage | Valeur | Pourquoi |
|---|---|---|
| "chunk_size" | 512 tokens | environ 2 ou 3 paragraphes : assez pour garder une idée complète, assez court pour que le vecteur reste précis |
| "chunk_overlap" | 64 tokens | environ 12 % : la fin d'un morceau est répétée au début du suivant, pour ne pas perdre le sens d'une phrase coupée à la frontière |

Ce sont des valeurs de départ courantes, pas des valeurs optimales. Les comparer sur les questions d'évaluation est une des améliorations prévues au TP 20.

Les métadonnées de chaque document sont recopiées dans chacun de ses morceaux : chaque extrait sait d'où il vient.


In [75]:
# splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
# nodes = splitter.get_nodes_from_documents(documents)

# print(len(nodes), "morceaux au total")
# nodes_per_source = Counter(node.metadata["source_id"] for node in nodes)
# for source_id, count in nodes_per_source.items():
#     print(f"  {source_id:13} {count:4} morceaux")

In [76]:
# # Contrôle : un morceau, puis la frontière entre ce morceau et le suivant
# node = nodes[20]
# print(node.metadata["source_id"], "|", len(node.text), "caractères")
# print(node.text)
# print()
# print("=== fin du morceau 20 ===")
# print(nodes[20].text[-200:])
# print("=== début du morceau 21 ===")
# print(nodes[21].text[:200])

### 3.5. Le stockage dans Chroma

On n'indexe pas notre extraction des sections 3.3 et 3.4, mais le corpus d'un collègue, déjà nettoyé et découpé : un dossier par source dans "data/corpus/", un fichier "chunk_N.txt" par morceau. Les sources écartées ont été sorties du dossier : "data/corpus/" contient exactement ce qui est indexé.

Les morceaux étant déjà découpés, chaque fichier devient directement un nœud ("TextNode"), sans passer par "SentenceSplitter". Son identifiant suit le format "doc:{source}#{numéro}", qui servira à citer la source. Les métadonnées ne sont pas vectorisées : seul le texte du morceau l'est.

Chroma est utilisé en mode persistant ("PersistentClient") : la base est un dossier, "data/index_rag/", qui contient un fichier SQLite (textes, métadonnées) et l'index HNSW des vecteurs. La collection est supprimée puis recréée à chaque exécution. Sinon, Chroma ignorerait sans prévenir les morceaux dont l'identifiant existe déjà (un morceau modifié garderait son ancien texte), et garderait ceux d'une source retirée du corpus. La distance choisie est la distance cosinus, comme dans l'exemple de la section 3.2.


In [77]:
corpus_nodes = []
for folder in sorted(CORPUS_DIR.iterdir()):
    if not folder.is_dir():  # on saute les fichiers isolés (.md, .pdf, .zip) : seuls les dossiers de morceaux comptent
        continue
    for chunk_path in folder.glob("chunk_*.txt"):
        number = int(chunk_path.stem.split("_")[1])  # "chunk_12" donne 12
        node = TextNode(
            id_=f"doc:{folder.name}#{number}",
            text=chunk_path.read_text(encoding="utf-8"),
            metadata={"source": folder.name, "chunk": number},
            excluded_embed_metadata_keys=["source", "chunk"],  # seul le texte est vectorisé
        )
        corpus_nodes.append(node)

sources = Counter(node.metadata["source"] for node in corpus_nodes)
print(len(corpus_nodes), "morceaux lus dans", len(sources), "sources")

405 morceaux lus dans 25 sources


In [78]:
INDEX_DIR = DATA_DIR / "index_rag"
COLLECTION_NAME = "corpus_nutrition"

client = chromadb.PersistentClient(path=str(INDEX_DIR))

# On repart d'une collection vide : aucun morceau obsolète ne reste dans la base
existing = [collection.name for collection in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
collection = client.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(corpus_nodes, storage_context=storage_context)  # vectorise et range les morceaux

print(collection.count(), "vecteurs dans Chroma")

405 vecteurs dans Chroma


In [79]:
# Contrôle : 5 morceaux tels qu'ils sont rangés dans Chroma
sample = collection.get(limit=5, include=["embeddings", "documents"])

for chunk_id, text, vector in zip(sample["ids"], sample["documents"], sample["embeddings"]):
    print(chunk_id)
    print("  texte    :", repr(text[:80]))
    print("  vecteur  :", len(vector), "valeurs, les 6 premières :", np.round(vector[:6], 3))
    print("  longueur :", round(float(np.linalg.norm(vector)), 3))

doc:2602_snanc#1
  texte    : 'L’agriculture et l’alimentation en France \n\nL’agriculture est un secteur économi'
  vecteur  : 1024 valeurs, les 6 premières : [-0.097 -0.006 -0.006  0.041  0.086  0.   ]
  longueur : 1.0
doc:2602_snanc#10
  texte    : 'Promues dans le cadre du 4ème PNNS, ces recommandations encouragent notamment un'
  vecteur  : 1024 valeurs, les 6 premières : [-0.135  0.022 -0.009  0.076  0.09  -0.004]
  longueur : 1.0
doc:2602_snanc#100
  texte    : 'La part de produits consommables qui sont gaspillés tout au long de la chaîne al'
  vecteur  : 1024 valeurs, les 6 premières : [-0.078  0.012 -0.009 -0.01   0.028 -0.025]
  longueur : 1.0
doc:2602_snanc#101
  texte    : 'Les orientations de la politique de l’alimentation et de la nutrition à horizon '
  vecteur  : 1024 valeurs, les 6 premières : [-0.085 -0.057 -0.006  0.044  0.088 -0.02 ]
  longueur : 1.0
doc:2602_snanc#102
  texte    : 'OBJECTIFS GENERAUX DE LA SNANC POUR 2030  • Diminuer de 30 %, par rapport à 2015'
  

### 3.6. La recherche

"as_retriever" transforme l'index en moteur de recherche : il vectorise la question (avec la consigne "Instruct:" de la section 3.2), demande à Chroma les "similarity_top_k" morceaux les plus proches, et les rend du plus proche au moins proche. Aucune réponse n'est rédigée.

Attention au score. Chroma rend une distance cosinus (0 = même direction), et le connecteur LlamaIndex la transforme en "exp(-distance)". Pour retrouver la similarité cosinus de la section 3.2 (1 - distance), on calcule d'abord la distance avec "-log(score)". Le classement est le même dans les deux cas, seule l'échelle change.


In [80]:
retriever = index.as_retriever(similarity_top_k=3)


def show_results(question):
    """Affiche les morceaux trouvés pour une question, avec leur score et leur source."""
    print("Question :", question)
    for result in retriever.retrieve(question):
        distance = -np.log(result.score)  # le connecteur rend exp(-distance)
        similarity = 1 - distance
        print(f"  {similarity:.2f}  {result.node.metadata['source']} #{result.node.metadata['chunk']}")
        print("        ", repr(result.node.text[:120]))
    print()


show_results("Combien de charcuterie peut-on manger par semaine ?")
show_results("Le Nutri-Score est-il obligatoire sur les emballages ?")

Question : Combien de charcuterie peut-on manger par semaine ?
  0.87  DT05-217-25B 32p_148x210_MANGERBOUGER 50 petites astuces_sept 2025_vf #51
         'Réduire la charcuterie. Limitez la charcuterie à maximum 150 g par semaine. Cela correspond à environ 3 tranches de jamb'
  0.84  Réduire la charcuterie  Manger Bouger #1
         'La charcuterie est le plus souvent grasse et salée, il est donc conseillé d’en limiter la consommation et de privilégier'
  0.68  Réduire la viande  porc, bœuf, veau, mouton, agneau, abats  Manger Bouger #1
         '500g de viande hors volaille par semaine, cela correspond à environ 3 ou 4 steaks. Pour instaurer de bonnes habitudes al'

Question : Le Nutri-Score est-il obligatoire sur les emballages ?
  0.86  dp_Nutri-Score_evolutions_20250625 #7
         'Qu’est-ce que le Nutri-Score ?\n\nUn logo nutritionnel pensé pour être facilement compris par les consommateurs\n\nMis en pl'
  0.86  DT05-115-24B T_NUTRISCORE_version sans reperes_juillet25_web_2 #1
  

### 3.7. Les fiches produits

Les produits sont des données structurées (une ligne, des colonnes), alors que le RAG cherche dans du texte. On transforme donc chaque produit en une fiche texte, à partir du modèle "docs/template/product_card_light.md".

Les noms entre accolades du modèle sont exactement les noms des colonnes du parquet nettoyé. On les lit directement dans le modèle pour construire la requête DuckDB, sans les recopier à la main. Si on ajoute un champ au modèle, la requête suit toute seule.

Le code-barres n'est pas dans le modèle : il ne porte aucun sens et ferait du bruit dans le vecteur. Il est ajouté à part dans la requête, parce qu'il sert d'identifiant ("fiche:{code}") et de métadonnée.

On travaille sur un échantillon de 1000 produits, car l'indexation complète prendrait plusieurs heures. Les produits sans nom ou sans Nutri-Score sont écartés.

Étapes : 1) lire le modèle et l'échantillon, 2) remplir les fiches et les écrire dans "data/product_card/", 3) les indexer dans une collection Chroma à part, 4) tester la recherche.

In [81]:
PRODUCT_CARD_TEMPLATE = (ROOT / "docs" / "template" / "product_card_light.md").read_text(encoding="utf-8")
CARD_COLUMNS = re.findall(r"\{(\w+)\}", PRODUCT_CARD_TEMPLATE)  # "{fat_100g}" donne "fat_100g"
print(CARD_COLUMNS)

clean_path = find_input(CLEAN_DIR, CLEAN_PREFIX, ".parquet")  # version datée la plus récente du parquet nettoyé
print("Fichier lu :", clean_path.name)

# Le filtre est dans la sous-requête : sinon DuckDB tire les 1000 lignes avant de filtrer et il en reste moins
query = f"""
    SELECT * FROM (
        SELECT code, {", ".join(CARD_COLUMNS)}
        FROM '{clean_path.as_posix()}'
        WHERE product_name IS NOT NULL AND nutriscore_grade IS NOT NULL
    )
    USING SAMPLE reservoir(1000 ROWS) REPEATABLE (42)
"""
products = duckdb.sql(query).df()  # REPEATABLE (42) : le même tirage à chaque exécution
print(len(products), "produits")
print(products.isna().sum())

['product_name', 'food_groups_tag_1', 'food_groups_tag_2', 'energy_kcal_100g', 'fat_100g', 'saturated_fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'nutriscore_grade']
Fichier lu : products_clean_20261116.parquet
1000 produits
code                    0
product_name            0
food_groups_tag_1      78
food_groups_tag_2      80
energy_kcal_100g        1
fat_100g                1
saturated_fat_100g      1
carbohydrates_100g      2
sugars_100g             3
fiber_100g            487
proteins_100g           1
salt_100g               0
nutriscore_grade        0
dtype: int64


In [82]:
CARD_DIR = DATA_DIR / "product_card"


def clean_value(value):
    """Rend une valeur du parquet prête à être écrite dans la fiche."""
    if pd.isna(value):
        return "non renseigné"
    if isinstance(value, str):
        return value.removeprefix("en:")  # "en:sugary-snacks" donne "sugary-snacks"
    return f"{value:g}"  # 18.200001 donne "18.2", 345.0 donne "345"


def make_card(row):
    """Remplit le modèle de fiche avec les valeurs d'un produit."""
    values = {}
    for column in CARD_COLUMNS:
        values[column] = clean_value(row[column])
    return PRODUCT_CARD_TEMPLATE.format(**values)


# On vide le dossier : il doit contenir exactement l'échantillon, pas les fiches d'un tirage précédent
CARD_DIR.mkdir(exist_ok=True)
for old_card in CARD_DIR.glob("*.md"):
    old_card.unlink()

for row in products.to_dict("records"):  # une ligne devient un dictionnaire {colonne: valeur}
    card_path = CARD_DIR / f"{row['code']}.md"
    card_path.write_text(make_card(row), encoding="utf-8")

print(len(list(CARD_DIR.glob("*.md"))), "fiches écrites dans", CARD_DIR.name)
print(make_card(products.iloc[1]))

1000 fiches écrites dans product_card
# Fiche produit — Spaghettis complets Bio Italie

**Catégorie :** cereals-and-potatoes  
**Sous-catégorie :** cereals

## Informations nutritionnelles

Valeurs pour 100 g ou 100 ml :

| Nutriment | Valeur |
|---|---|
| Énergie | 345 kcal |
| Matières grasses | 2.7 g |
| Acides gras saturés | 0.6 g |
| Glucides | 65 g |
| Sucres | 5 g |
| Fibres alimentaires | 5.2 g |
| Protéines | 12 g |
| Sel | 0 g |

## Indicateurs nutritionnels

**Nutri-Score :** b



In [83]:
CARD_COLLECTION_NAME = "fiches_produits"

card_nodes = []
for row in products.to_dict("records"):
    card_path = CARD_DIR / f"{row['code']}.md"
    node = TextNode(
        id_=f"fiche:{row['code']}",                  # id stable : le même produit garde le même id
        text=card_path.read_text(encoding="utf-8"),  # le texte vient du fichier écrit à l'étape 2
        metadata={
            "code": row["code"],
            "nutriscore_grade": row["nutriscore_grade"],  # servira à filtrer ("uniquement des produits notés A")
            "food_group": clean_value(row["food_groups_tag_1"]),
            "source": "Open Food Facts",
        },
        excluded_embed_metadata_keys=["code", "nutriscore_grade", "food_group", "source"],  # seul le texte est vectorisé
    )
    card_nodes.append(node)

# Même client que la section 3.5 : une seule base, deux collections séparées
existing = [collection.name for collection in client.list_collections()]
if CARD_COLLECTION_NAME in existing:
    client.delete_collection(CARD_COLLECTION_NAME)
card_collection = client.create_collection(CARD_COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

card_store = ChromaVectorStore(chroma_collection=card_collection)
card_storage_context = StorageContext.from_defaults(vector_store=card_store)
card_index = VectorStoreIndex(card_nodes, storage_context=card_storage_context)  # environ 30 s pour 1000 fiches

print(card_collection.count(), "fiches dans Chroma")
print("Collections :", [collection.name for collection in client.list_collections()])

1000 fiches dans Chroma
Collections : ['corpus_nutrition', 'fiches_produits']


In [84]:
card_retriever = card_index.as_retriever(similarity_top_k=3)


def show_cards(question):
    """Affiche les fiches trouvées pour une question : similarité, note et nom du produit."""
    print("Question :", question)
    for result in card_retriever.retrieve(question):
        similarity = 1 + np.log(result.score)  # même conversion qu'en 3.6 : 1 - distance
        name = result.node.text.splitlines()[0].removeprefix("# Fiche produit — ")  # le titre de la fiche
        print(f"  {similarity:.2f}  {result.node.metadata['nutriscore_grade']}  {name}")
    print()


show_cards("des spaghettis complets bio")
show_cards("un yaourt nature")
show_cards("une pâte à tartiner moins sucrée")

Question : des spaghettis complets bio
  0.76  b  Spaghettis complets Bio Italie
  0.57  c  Pain complet bio
  0.56  a  Tagliatelles bio au piment d'espelette aop

Question : un yaourt nature
  0.66  b  Yaourt brassé Nature
  0.62  d  Yaourt à boire vanille
  0.62  c  Yaourt

Question : une pâte à tartiner moins sucrée
  0.55  e  Pâte à tartiner Noisettes et Cacao Bio et équitable
  0.55  d  Tarte aux myrtilles
  0.55  e  Pâte à tartiner chocolat speculoos



La recherche par similarité retrouve bien un produit décrit par son nom ou son type ("spaghettis complets bio", "yaourt nature"). Elle ne comprend pas une comparaison comme "moins sucrée" : elle trouve des pâtes à tartiner, mais pas les moins sucrées, parce qu'elle compare des sens et pas des nombres.

Pour ce genre de demande, la recherche dans les fiches ("rag_search_products", section 3.9) devra être complétée : un filtre exact sur la note grâce aux métadonnées ("nutriscore_grade"), ou un tri sur les valeurs nutritionnelles avec une requête SQL.

### 3.8. Une mesure simple de la recherche

Regarder quelques résultats à l'œil ne suffit pas pour dire que la recherche fonctionne. On écrit donc des questions dont on connaît la réponse, et on compte combien de fois une bonne réponse apparaît dans les 3 premiers résultats (le "taux de réussite au top 3").

Chaque collection est mesurée séparément, avec ses propres questions, car chaque outil de l'agent n'utilisera que l'une des deux :

- pour le corpus, la réponse attendue est le dossier de la source qui contient la réponse,
- pour les fiches, ce sont les codes-barres des produits qui conviennent.

Une question peut avoir plusieurs réponses acceptables (5 fiches de comté conviennent), mais on n'ajoute que les sources qui répondent vraiment, sinon le score est gonflé. C'est pour cette raison que la brochure "50 petites astuces" n'est pas attendue pour la question sur les enfants : elle parle des enfants, mais ne dit pas combien de produits sucrés leur donner.

On mesure ici si la recherche trouve la bonne source, pas si la réponse finale de l'agent est juste.

In [87]:
questions_corpus = [
    ("Combien de produits sucrés je peux donner à mon enfant en bas âge ?",
     ["Réduire les boissons sucrées, aliments gras, sucrés, salés et ultra-transformés  Manger Bouger"]),
    ("Quelle quantité de poisson donner à un enfant de 5 ans ?",
     ["Aller vers les poissons gras et maigres en alternance  Manger Bouger"]),
    ("Je bois du vin, quelle est la limite à ne pas dépasser sur une semaine ?",
     ["Réduire l'alcool  Manger Bouger"]),
]

# Codes valables pour l'échantillon de "products_clean_20261116.parquet" (graine 42)
questions_cards = [
    ("Je cherche un fromage de comté",
     ["0202881037261", "2165748004371", "2454495034770", "2480184024807", "2619022020934"]),
    ("du saumon fumé de Norvège en tranches",
     ["3760337630833"]),
    ("une compote de pommes light",
     ["20446956"]),
]


def hit_rate(retriever, questions, key):
    """Part des questions dont au moins une réponse attendue est dans les résultats (top 3)."""
    hits = 0
    for question, expected in questions:
        found = []
        for result in retriever.retrieve(question):
            found.append(result.node.metadata[key])  # "source" pour le corpus, "code" pour les fiches
        hit = False
        for answer in expected:
            if answer in found:
                hit = True
        if hit:
            hits += 1
        print("OK   " if hit else "RATÉ ", question)
        for value in found:
            print("        ", value)
    return hits / len(questions)


print("Corpus :", hit_rate(retriever, questions_corpus, "source"))
print()
print("Fiches :", hit_rate(card_retriever, questions_cards, "code"))

OK    Combien de produits sucrés je peux donner à mon enfant en bas âge ?
         Réduire les boissons sucrées, aliments gras, sucrés, salés et ultra-transformés  Manger Bouger
         Réduire les boissons sucrées, aliments gras, sucrés, salés et ultra-transformés  Manger Bouger
         Augmenter les fruits à coque non salés noix, noisettes, amandes, etc.  Manger Bouger
OK    Quelle quantité de poisson donner à un enfant de 5 ans ?
         Aller vers les poissons gras et maigres en alternance  Manger Bouger
         Réduire la viande  porc, bœuf, veau, mouton, agneau, abats  Manger Bouger
         DT05-217-25B 32p_148x210_MANGERBOUGER 50 petites astuces_sept 2025_vf
OK    Je bois du vin, quelle est la limite à ne pas dépasser sur une semaine ?
         Réduire l'alcool  Manger Bouger
         DT05-217-25B 32p_148x210_MANGERBOUGER 50 petites astuces_sept 2025_vf
         Réduire l'alcool  Manger Bouger
Corpus : 1.0

OK    Je cherche un fromage de comté
         2454495034770
       

### 3.9. Les fonctions de recherche

Les outils de l'agent (section 4) n'appelleront pas les retrievers directement, mais deux fonctions qui rendent toujours le même format, celui du TP 19 : une liste de dictionnaires avec "id", "score", "texte" et "metadonnees". L'agent reçoit ainsi des résultats simples à lire et à citer, quelle que soit la collection.

- "rag_search_docs(question, k)" cherche dans le corpus nutrition.
- "rag_search_products(question, grade, k)" cherche dans les fiches produits. Si "grade" est donné, un filtre exact sur la métadonnée "nutriscore_grade" ne garde que les fiches de cette note, avant de chercher les plus proches ("MetadataFilters").

Le préfixe "rag_" les distingue de "search_product" (TP 18), qui interroge la base à partir d'un code-barres ou d'un nom exact. Les fonctions "rag_" cherchent par similarité de sens dans Chroma.

Le score est la similarité cosinus (1 - distance), recalculée comme en 3.6 et arrondie, pour que l'agent et le journal voient la même échelle que dans la section 3.2.

In [92]:
def to_result(result):
    """Transforme un résultat LlamaIndex au format du TP 19 : id, score, texte, métadonnées."""
    distance = -np.log(result.score)  # le connecteur rend exp(-distance), voir 3.6
    return {
        "id": result.node.id_,
        "score": round(1 - distance, 3),  # similarité cosinus : 1 = même sens
        "texte": result.node.text,
        "metadonnees": result.node.metadata,
    }


def rag_search_docs(question, k=3):
    """Cherche les k morceaux du corpus nutrition les plus proches de la question."""
    results = []
    for result in index.as_retriever(similarity_top_k=k).retrieve(question):
        results.append(to_result(result))
    return results


def rag_search_products(question, grade=None, k=3):
    """Cherche les k fiches produits les plus proches, en ne gardant que la note "grade" si elle est donnée."""
    filters = None
    if grade is not None:
        # filtre exact sur la métadonnée, les notes sont stockées en minuscules ("a" à "e")
        filters = MetadataFilters(filters=[MetadataFilter(key="nutriscore_grade", value=grade.lower())])
    results = []
    for result in card_index.as_retriever(similarity_top_k=k, filters=filters).retrieve(question):
        results.append(to_result(result))
    return results


for result in rag_search_docs("Combien de charcuterie peut-on manger par semaine ?"):
    print(result["score"], result["id"], repr(result["texte"][:60]))
print()

for question, grade in [("un yaourt nature", None), ("un yaourt nature", "A"), ("une pâte à tartiner", "A")]:
    print(question, "| note :", grade)
    for result in rag_search_products(question, grade=grade):
        print("  ", result["score"], result["metadonnees"]["nutriscore_grade"], result["texte"].splitlines()[0])

0.867 doc:DT05-217-25B 32p_148x210_MANGERBOUGER 50 petites astuces_sept 2025_vf#51 'Réduire la charcuterie. Limitez la charcuterie à maximum 150'
0.839 doc:Réduire la charcuterie  Manger Bouger#1 'La charcuterie est le plus souvent grasse et salée, il est d'
0.678 doc:Réduire la viande  porc, bœuf, veau, mouton, agneau, abats  Manger Bouger#1 '500g de viande hors volaille par semaine, cela correspond à '

un yaourt nature | note : None
   0.662 b # Fiche produit — Yaourt brassé Nature
   0.622 d # Fiche produit — Yaourt à boire vanille
   0.617 c # Fiche produit — Yaourt
un yaourt nature | note : A
   0.547 a # Fiche produit — Eau de source naturelle
   0.523 a # Fiche produit — VITTEL eau minérale naturelle 6 x 1,5L
   0.505 a # Fiche produit — ROZANA Eau Minérale Naturelle Pétillante 0.50 L
une pâte à tartiner | note : A
   0.544 a # Fiche produit — Pâtes Torsade qualité supérieure paquet cello de 500g
   0.504 a # Fiche produit — Pane intégrale
   0.501 a # Fiche produit — Pâtes lam

Le filtre fonctionne : avec la note A, toutes les fiches rendues sont notées a. Mais il cache un piège. L'échantillon ne contient aucun yaourt nature ni aucune pâte à tartiner notés A, et la recherche rend quand même 3 résultats : des eaux minérales et des pâtes alimentaires. Le retriever rend toujours les k plus proches parmi ce qui reste après le filtre, même quand aucun n'est pertinent.

Le score le montre : environ 0,5 pour ces résultats hors sujet, contre 0,62 à 0,66 pour les vrais yaourts sans filtre. L'agent devra donc vérifier que les fiches rendues correspondent bien à la demande, et savoir répondre « je n'ai pas trouvé de yaourt noté A » plutôt que de proposer de l'eau.

## 4. Les outils de l'agent

### 4.1. La recherche exacte dans le catalogue : "search_product"

Cette fonction suit le contrat "chercher_produit" du TP 18, imposé à toutes les équipes : mêmes arguments ("nom", "code", "limite") et même forme de résultat ("statut", "total", "produits"). Seul le nom de la fonction est en anglais (écart à signaler à la revue croisée).

Elle cherche dans le parquet nettoyé avec DuckDB, par correspondance exacte, contrairement à "rag_search_products" qui cherche par similarité de sens :

- par code-barres exact, qui prime sur le nom si les deux sont donnés,
- ou par nom ou partie du nom, sans tenir compte de la casse ni des accents ("strip_accents" et "lower").

Les règles du contrat :

| Règle | Comment elle est appliquée |
|---|---|
| les arguments viennent du modèle, ils ne sont pas fiables | vérifiés avant toute requête, une erreur est rendue sous la forme {"erreur": ...} au lieu d'une exception |
| pas d'injection SQL | le nom et le code passent en paramètres ("?"), jamais collés dans la requête |
| statut | "ok" pour un seul produit, "ambigu" pour plusieurs (le modèle doit faire préciser), "introuvable" pour aucun |
| valeur manquante | None, jamais 0, et "donnees_incompletes" vaut vrai si la note ou un nutriment manque |

Quand plusieurs produits correspondent, les fiches les plus complètes ("completeness") sont rendues en premier.

In [ ]:
NUTRIENTS = {  # clé du contrat TP 18 : colonne du parquet nettoyé
    "kcal": "energy_kcal_100g",
    "lipides": "fat_100g",
    "satures": "saturated_fat_100g",
    "sucres": "sugars_100g",
    "sel": "salt_100g",
    "fibres": "fiber_100g",
    "proteines": "proteins_100g",
}


def to_value(value):
    """Rend None pour une valeur manquante (jamais 0), sinon la valeur arrondie."""
    if pd.isna(value):
        return None
    return round(float(value), 2)


def make_fiche(row):
    """Construit la fiche du contrat TP 18 à partir d'une ligne du parquet nettoyé."""
    nutriments = {}
    for key, column in NUTRIENTS.items():
        nutriments[key] = to_value(row[column])
    grade = None if pd.isna(row["nutriscore_grade"]) else row["nutriscore_grade"].upper()
    incomplete = grade is None or None in nutriments.values()
    return {
        "code": row["code"],
        "nom": row["product_name"],
        "nutriscore": grade,
        "nutriments": nutriments,
        "donnees_incompletes": incomplete,
    }


def search_product(nom=None, code=None, limite=3):
    """Cherche un produit par code-barres exact ou par nom (contrat chercher_produit du TP 18)."""
    # Les arguments viennent du modèle : on les vérifie avant toute requête
    if not nom and not code:
        return {"erreur": "il faut au moins un nom ou un code-barres"}
    if not isinstance(limite, int) or not 1 <= limite <= 5:
        return {"erreur": "limite doit être un entier de 1 à 5"}

    if code:  # le code prime sur le nom
        where = "code = ?"
        params = [str(code)]
    else:     # nom ou partie du nom, sans tenir compte de la casse ni des accents
        where = "strip_accents(lower(product_name)) LIKE '%' || strip_accents(lower(?)) || '%'"
        params = [nom]

    # Le "?" est remplacé par DuckDB lui-même : la valeur reste une donnée, jamais du code SQL
    source = f"FROM '{clean_path.as_posix()}' WHERE {where}"
    total = duckdb.execute(f"SELECT count(*) {source}", params).fetchone()[0]
    columns = ", ".join(["code", "product_name", "nutriscore_grade", *NUTRIENTS.values()])
    rows = duckdb.execute(
        f"SELECT {columns} {source} ORDER BY completeness DESC NULLS LAST LIMIT {limite}", params
    ).df()

    fiches = []
    for row in rows.to_dict("records"):
        fiches.append(make_fiche(row))

    if total == 0:
        statut = "introuvable"
    elif total == 1:
        statut = "ok"
    else:
        statut = "ambigu"
    return {"statut": statut, "total": total, "produits": fiches}


tests = [
    {"code": "3017620422003"},                   # produit connu : ok
    {"nom": "comte AOP 6 mois"},                 # sans accent ni majuscule : ambigu
    {"nom": "xyzproduitinexistant"},             # introuvable
    {},                                          # ni nom ni code : erreur
    {"nom": "yaourt", "limite": 9},              # limite hors bornes : erreur
    {"code": "x' OR '1'='1"},                    # tentative d'injection SQL : rien trouvé
]
for arguments in tests:
    result = search_product(**arguments)
    print(arguments)
    print("   ", result.get("statut", result.get("erreur")), "| total :", result.get("total"))
    for fiche in result.get("produits", []):
        print("    ", fiche["code"], fiche["nom"], fiche["nutriscore"], fiche["nutriments"])